# MediaPlan Optimizer - демонстрация

Здесь я прохожу три основных сценария через тот же `optimize_media_plan`, который использует приложение: фиксированный бюджет, подбор бюджета под KPI и недостижимая цель.
Основной план считаю в Calendar-aware, а Uniform оставляю для сравнения.
Каталог содержит восемь абстрактных инвентарей: это сохранённая конфигурация, полученная из диапазонов с seed 42.
Для запуска сверху вниз нужны зависимости из `requirements.txt` и Python-окружение проекта.

In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "mediaplan_optimizer").is_dir():
    raise RuntimeError("Откройте notebook из корня проекта или папки notebooks.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from mediaplan_optimizer import OptimizationRequest, PlanningMode, load_catalog, optimize_media_plan
from mediaplan_optimizer.schemas import DEFAULT_CALENDAR_START
from ui_text import feasibility_text

pd.options.display.float_format = "{:,.2f}".format
catalog = load_catalog()
channels = pd.DataFrame([{
    "Инвентарь": channel.name,
    "Тип": channel.channel_type,
    "Базовый CPM, ₽": channel.base_cpm_rub,
    "Дневная ёмкость показов": channel.daily_capacity,
    "Дневной лимит расходов, ₽": channel.max_daily_spend,
    "Базовый CTR": channel.base_ctr,
    "Базовый CR": channel.base_cr,
} for channel in catalog.values()])
display(channels.style.format({
    "Базовый CPM, ₽": "{:,.2f}",
    "Дневная ёмкость показов": "{:,.0f}",
    "Дневной лимит расходов, ₽": "{:,.0f}",
    "Базовый CTR": "{:.2%}",
    "Базовый CR": "{:.2%}",
}).hide(axis="index"))

Инвентарь,Тип,"Базовый CPM, ₽",Дневная ёмкость показов,"Дневной лимит расходов, ₽",Базовый CTR,Базовый CR
Social Network 1,media,223.77,"2,218,536","1,290,644",1.57%,3.75%
Social Network 2,media,268.30,"1,836,692","1,203,807",1.40%,1.82%
Social Network 3,media,180.94,"1,291,540","903,705",1.49%,3.02%
Programmatic,media,149.14,"4,490,484","2,012,401",0.35%,1.78%
Marketplace 1,media,464.64,"912,863","1,204,451",2.14%,4.88%
Marketplace 2,media,433.46,"861,009","752,768",2.64%,3.85%
Marketplace 3,media,354.07,"529,239","689,619",1.63%,3.43%
SMS,sms,351.75,"396,112","347,691",3.48%,2.91%


## Type A: 1,2 млн рублей на 21 день

Хочу получить максимум конверсий. Оставляю все каналы и явно выбираю Calendar-aware: начало 21 сентября 2026 года, seed календарных профилей 42.

Сначала посмотрю общий прогноз и каналы, которым выделены деньги, затем - расходы по дням. Это прогноз при параметрах каталога, не измеренные результаты кампании.

In [2]:
request_a = OptimizationRequest(
    task_type="A",
    planning_mode="calendar_aware",
    budget=1_200_000,
    horizon_days=21,
    objective_metric="conversions",
    included_channels=list(catalog),
    start_date=DEFAULT_CALENDAR_START,
    calendar_profile_seed=42,
)
result_a = optimize_media_plan(request_a, catalog)

display(pd.DataFrame([{
    "Расход, ₽": result_a.summary.spend,
    "Конверсии": result_a.summary.conversions,
    "Клики": result_a.summary.clicks,
    "CPA, ₽": result_a.summary.cpa,
    "Эффективный CPM, ₽": result_a.summary.cpm,
    "Неиспользованный бюджет, ₽": result_a.summary.unspent_budget,
}]))
channels = pd.DataFrame([{
    "Инвентарь": row.channel,
    "Расход, ₽": row.spend,
    "Конверсии": row.conversions,
    "CPA, ₽": row.cpa,
} for row in result_a.channel_metrics if row.spend > 0])
display(channels.sort_values("Расход, ₽", ascending=False).reset_index(drop=True))

,"Расход, ₽",Конверсии,Клики,"CPA, ₽","Эффективный CPM, ₽","Неиспользованный бюджет, ₽"
0,"1,200,000.00","3,077.08","92,684.57",389.98,263.66,0.00


,Инвентарь,"Расход, ₽",Конверсии,"CPA, ₽"
0,Social Network 1,"660,810.67","1,669.78",395.75
1,SMS,"434,037.52","1,149.20",377.69
2,Social Network 3,"96,562.30",237.23,407.04
3,Marketplace 2,"8,589.50",20.86,411.70


In [3]:
daily = (
    pd.DataFrame([{"date": row.date, "spend": row.spend} for row in result_a.allocations])
    .groupby("date", as_index=False)["spend"].sum()
)
figure = px.bar(
    daily,
    x="date",
    y="spend",
    title="Calendar-aware: расходы по дням кампании",
    labels={"date": "Дата", "spend": "Расход, ₽"},
)
figure.update_layout(template="plotly_white", showlegend=False)
figure.update_xaxes(dtick="D2", tickformat="%d.%m")
figure.show(renderer="plotly_mimetype")

## Тот же бюджет: Calendar-aware и Uniform

Uniform считает дни одинаковыми и равномерно распределяет дневные расходы каждого канала. Calendar-aware выбирает расходы сразу по парам канал-день.

Ниже меняю только режим планирования. Разница в этом синтетическом сценарии небольшая; из неё нельзя делать вывод, что календарный режим всегда лучше.

In [4]:
request_uniform = request_a.model_copy(update={"planning_mode": PlanningMode.UNIFORM})
result_uniform = optimize_media_plan(request_uniform, catalog)

display(pd.DataFrame([{
    "Режим": mode,
    "Расход, ₽": result.summary.spend,
    "Конверсии": result.summary.conversions,
    "CPA, ₽": result.summary.cpa,
} for mode, result in [
    ("Calendar-aware", result_a),
    ("Uniform V1", result_uniform),
]]))

,Режим,"Расход, ₽",Конверсии,"CPA, ₽"
0,Calendar-aware,"1,200,000.00","3,077.08",389.98
1,Uniform V1,"1,200,000.00","3,069.98",390.88


## Type B: сколько денег нужно на 10 000 кликов?

Теперь решу обратную задачу: сервис ищет минимальный модельный бюджет, при котором цель достижима за 14 дней. Использую все каналы и те же настройки Calendar-aware.

In [5]:
request_b = OptimizationRequest(
    task_type="B",
    planning_mode="calendar_aware",
    target_value=10_000,
    horizon_days=14,
    objective_metric="clicks",
    included_channels=list(catalog),
    start_date=DEFAULT_CALENDAR_START,
    calendar_profile_seed=42,
)
result_b = optimize_media_plan(request_b, catalog)

display(pd.DataFrame([{
    "Цель, клики": request_b.target_value,
    "Прогноз, клики": result_b.summary.achieved_target,
    "Минимальный модельный бюджет, ₽": result_b.summary.spend,
    "CPC, ₽": result_b.summary.cpc,
}]))

,"Цель, клики","Прогноз, клики","Минимальный модельный бюджет, ₽","CPC, ₽"
0,"10,000.00","10,000.01","103,937.97",10.39


In [6]:
display(Markdown(
    "## Если цель недостижима\n\n"
    "Запрошу триллион конверсий за один день, снова в Calendar-aware. "
    "Здесь важно не огромное число само по себе: вместо пустого плана "
    "сервис оценивает ёмкость модели и объясняет, что можно изменить."
))
request_infeasible = OptimizationRequest(
    task_type="B",
    planning_mode="calendar_aware",
    target_value=1_000_000_000_000,
    horizon_days=1,
    objective_metric="conversions",
    included_channels=list(catalog),
    start_date=DEFAULT_CALENDAR_START,
    calendar_profile_seed=42,
)
result_infeasible = optimize_media_plan(request_infeasible, catalog)
feasibility = result_infeasible.feasibility

display(pd.DataFrame([{
    "Цель, конверсии": feasibility.requested_value,
    "Максимум за день": feasibility.maximum_achievable,
    "Недостающие конверсии": feasibility.target_gap,
}]))
explanation, recommendations = feasibility_text(
    feasibility, request_infeasible.objective_metric.value, "ru",
    horizon_days=request_infeasible.horizon_days,
)
display(Markdown("**Причина:** " + explanation.replace("—", "-")))
display(Markdown("**Рекомендации:**\n\n" + "\n".join("- " + item for item in recommendations)))

## Если цель недостижима

Запрошу триллион конверсий за один день, снова в Calendar-aware. Здесь важно не огромное число само по себе: вместо пустого плана сервис оценивает ёмкость модели и объясняет, что можно изменить.

,"Цель, конверсии",Максимум за день,Недостающие конверсии
0,"1,000,000,000,000.00","3,955.23","999,999,996,044.77"


**Причина:** Запрошено 1,000,000,000,000.00 конверсий, но практический максимум для выбранных каналов и срока - 3,955.23.

**Рекомендации:**

- Снизьте цель при сроке 1 дней до примерно 3,950.00 конверсий.

## Что получилось

Через один публичный сервис получились все три обязательных сценария: фиксированный бюджет, минимальный бюджет под KPI и понятное объяснение недостижимой цели. Calendar-aware использует те же кривые отклика, но принимает решение сразу по парам канал-день.

[Интерактивная версия проекта](https://shad-capstone-mediaplan-optimizer.streamlit.app/) · [Математика и ограничения модели](../docs/MATHEMATICAL_MODEL.md).

Работа с фактом, мониторинг и адаптивные стратегии оставлены за пределами этого notebook и доступны в приложении.